# _lib/derivation_helpers — Gold wide-table afleidingshulpfuncties

Pure hulpfuncties voor de tijd- en regelafgeleide kolommen op
`DATAMART.sales_lines_wide`.

**Importeer via `%run ./_lib/derivation_helpers` vanuit het Gold DLT-pipeline notebook.**

| Functie | Geeft terug | Gebruikt voor |
|---|---|---|
| `derive_order_date(col)` | `DateType` | `CAST(order_ts AS DATE)` |
| `derive_order_hour(col)` | `IntegerType` | `HOUR(order_ts)` (0–23) |
| `derive_order_day_of_week(col)` | `StringType` | `DATE_FORMAT(order_ts, 'EEEE')` (bijv. 'Monday') |
| `derive_order_year_month(col)` | `DateType` | `DATE_TRUNC('month', order_ts)` (eerste dag van de maand, DATE) |
| `derive_shift_duration_minutes(start_col, end_col)` | `IntegerType` | `(end_seconds - start_seconds) / 60` uit `'HH:mm:ss'`-strings |
| `derive_line_subtotal(qty_col, price_col)` | `DecimalType(38, 4)` | `quantity * unit_price` |

Alle functies retourneren een Spark `Column`-expressie — geen neveneffecten, geen Spark-context
afhankelijkheid in de functie-signaturen zelf (Column-objecten zijn lazy).

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql import Column


def derive_order_date(order_ts_col: Column) -> Column:
    """Converteert een TimestampType order_ts-kolom naar DateType.

    Example: 2024-03-15 14:30:00  ->  2024-03-15
    """
    return order_ts_col.cast("date")


def derive_order_hour(order_ts_col: Column) -> Column:
    """Extraheert het uur van de dag (0–23) uit een TimestampType order_ts-kolom.

    Geeft een IntegerType terug.  Maakt filtering op uur-bucket in dashboards
    en Genie mogelijk zonder herberekening bij elke query.

    Example: 2024-03-15 14:30:00  ->  14
    """
    return F.hour(order_ts_col).cast("int")


def derive_order_day_of_week(order_ts_col: Column) -> Column:
    """Extraheert de volledige dagnaam uit een TimestampType order_ts-kolom.

    Gebruikt Spark's EEEE locale-gevoelig formaatpatroon.  Geeft een StringType-waarde
    terug zoals 'Monday', 'Tuesday', enz.  De Databricks-cluster locale staat standaard
    op Engels.

    Example: 2024-03-15 (Friday)  ->  'Friday'
    """
    return F.date_format(order_ts_col, "EEEE")


def derive_order_year_month(order_ts_col: Column) -> Column:
    """Kapt een TimestampType order_ts-kolom af op de eerste dag van de maand.

    Geeft een DateType-waarde terug (eerste-van-de-maand DATE).  Maakt maand-niveau
    filtering en groepering in dashboards mogelijk zonder string-parsing.

    Example: 2024-03-15 14:30:00  ->  2024-03-01
    """
    return F.date_trunc("month", order_ts_col).cast("date")


def _hhmmss_to_seconds(hhmmss_col: Column) -> Column:
    """Converteert een 'HH:mm:ss' StringType-kolom naar seconden-sinds-middernacht als IntegerType.

    Silver slaat shift_start_time en shift_end_time op als 'HH:mm:ss'-strings.
    Deze hulpfunctie is intern aan derivation_helpers; aanroepers moeten
    derive_shift_duration_minutes gebruiken.

    Example: '10:30:45'  ->  37845
    """
    hh = F.split(hhmmss_col, ":").getItem(0).cast("int")
    mm = F.split(hhmmss_col, ":").getItem(1).cast("int")
    ss = F.split(hhmmss_col, ":").getItem(2).cast("int")
    return (hh * 3600 + mm * 60 + ss)


def derive_shift_duration_minutes(shift_start_col: Column, shift_end_col: Column) -> Column:
    """Berekent de shiftduur in hele minuten uit twee 'HH:mm:ss' StringType-kolommen.

    Silver's shift_start_time en shift_end_time zijn 'HH:mm:ss'-strings.
    Deze functie parseert beide naar seconden-sinds-middernacht en geeft het
    verschil in minuten terug, gecast naar IntegerType (integerdeling — de seconden-
    rest wordt weggegooid).

    Geeft NULL terug wanneer een van de invoeren NULL is (natuurlijke Spark NULL-propagatie).

    Example: shift_start='08:00:00', shift_end='10:30:00'  ->  150
    """
    start_seconds = _hhmmss_to_seconds(shift_start_col)
    end_seconds   = _hhmmss_to_seconds(shift_end_col)
    return ((end_seconds - start_seconds) / 60).cast("int")


def derive_line_subtotal(quantity_col: Column, unit_price_col: Column) -> Column:
    """Berekent de regelsubtotaal als quantity * unit_price, gecast naar DECIMAL(38, 4).

    Biedt een controlekolom ten opzichte van de Bronze `price`-kolom —
    analisten kunnen `line_subtotal` vergelijken met `price` om prijsafwijkingen
    te detecteren zonder de expressie bij elke query opnieuw te berekenen.

    Geeft NULL terug wanneer een van de invoeren NULL is (natuurlijke Spark NULL-propagatie).

    Example: quantity=3, unit_price=4.5000  ->  13.5000
    """
    return (quantity_col * unit_price_col).cast("decimal(38, 4)")


print(
    "derivation_helpers geladen: "
    "derive_order_date, derive_order_hour, derive_order_day_of_week, "
    "derive_order_year_month, derive_shift_duration_minutes, derive_line_subtotal"
)